In [ ]:
import gc
import torch

# 先清理一下上一个 notebook 可能残留的显存缓存。
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    print('GPU cache cleared.')
else:
    print('CUDA not available, skip GPU cache clear.')


# 下一课：CartPole + PPO

你已经学过：
- `REINFORCE`
- `REINFORCE + baseline`
- `Actor-Critic`
- `A2C`

这一课我们继续进入策略优化路线里非常重要的一种方法：`PPO`，也就是 `Proximal Policy Optimization`。

它之所以很重要，是因为它通常：
- 比简单策略梯度更稳
- 比很多复杂算法更容易实现
- 在实际项目和教学里都非常常见


## 1. 为什么还需要 PPO

前面的策略梯度方法有一个共同风险：

**一次更新如果走得太猛，策略可能突然变坏。**

比如：
- 某次 advantage 很大
- 梯度一步把动作概率改得太激进
- 下一轮策略就可能偏到很奇怪的方向

PPO 的核心思想就是：

**更新可以有，但别一下子把策略改太多。**


## 2. PPO 最核心的直觉

PPO 会比较：
- 旧策略下，这个动作的概率是多少
- 新策略下，这个动作的概率变成了多少

然后看这个比例：

$ratio = \frac{\pi_{new}(a|s)}{\pi_{old}(a|s)}$

如果这个比例变化太大，PPO 就会把更新“截住”，不让它继续走太猛。

这就是 PPO 里最著名的 `clipping` 思想。


## 3. PPO 的 clipped objective

PPO 的关键目标函数可以写成：

$L = min(ratio \cdot A, clip(ratio, 1-\epsilon, 1+\epsilon) \cdot A)$

你现在不需要死记公式，先抓住意思：

- 如果新旧策略差别不大，就正常更新
- 如果新策略已经偏得太远，就把更新截住

这样做的结果通常就是：

**训练更稳，不容易一步把策略搞坏。**


In [ ]:
import random
import warnings

import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical

plt.style.use('seaborn-v0_8-whitegrid')
np.set_printoptions(precision=3, suppress=True)

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)


In [ ]:
def pick_device():
    if torch.cuda.is_available():
        try:
            _ = torch.zeros(1, device='cuda')
            return torch.device('cuda')
        except Exception as e:
            warnings.warn(f'检测到 CUDA，但当前环境无法真正使用 GPU，已回退到 CPU。原因: {e}')
    return torch.device('cpu')


device = pick_device()
print('当前设备:', device)
if torch.cuda.is_available():
    print('检测到 CUDA 设备:', torch.cuda.get_device_name(0))


In [ ]:
env = gym.make('CartPole-v1')
state, info = env.reset(seed=42)
print('初始状态:', state)
print('状态维度:', env.observation_space.shape[0])
print('动作数量:', env.action_space.n)


In [ ]:
def to_tensor(x, device):
    return torch.tensor(x, dtype=torch.float32, device=device)


class ActorCriticNet(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim=128):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU()
        )
        self.policy_head = nn.Linear(hidden_dim, action_dim)
        self.value_head = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        features = self.shared(x)
        logits = self.policy_head(features)
        value = self.value_head(features)
        return logits, value


## 4. PPO 的训练流程

这节课我们采用一个教学友好的 PPO 版本：

1. 先用当前策略收集一批轨迹
2. 记录旧策略下动作的 `log_prob`
3. 计算 return 和 advantage
4. 用 PPO 的 clipped objective 做多轮更新

你要重点注意：

**PPO 不是边收集边立刻更新，而是先收集一批，再反复利用这一批数据训练几轮。**


In [ ]:
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.n

model = ActorCriticNet(state_dim, action_dim, hidden_dim=128).to(device)
optimizer = optim.Adam(model.parameters(), lr=3e-4)

gamma = 0.99
episodes = 300
max_steps = 500
ppo_epochs = 4
clip_eps = 0.2
value_coef = 0.5
entropy_coef = 0.01

episode_rewards = []
policy_loss_history = []
value_loss_history = []
entropy_history = []

for episode in range(episodes):
    state, info = env.reset()

    states = []
    actions = []
    rewards = []
    dones = []
    old_log_probs = []
    values = []
    total_reward = 0.0

    # 先采样一整条轨迹
    for step in range(max_steps):
        state_tensor = to_tensor(state, device).unsqueeze(0)
        logits, value = model(state_tensor)
        dist = Categorical(logits=logits)

        action = dist.sample()
        log_prob = dist.log_prob(action)

        next_state, reward, terminated, truncated, info = env.step(int(action.item()))
        done = terminated or truncated

        states.append(state)
        actions.append(int(action.item()))
        rewards.append(reward)
        dones.append(done)
        old_log_probs.append(log_prob.detach())
        values.append(value.squeeze(1).detach())

        total_reward += reward
        state = next_state

        if done:
            break

    # 计算 return
    returns = []
    G = 0.0
    for r in reversed(rewards):
        G = r + gamma * G
        returns.insert(0, G)

    states_tensor = to_tensor(np.array(states), device)
    actions_tensor = torch.tensor(actions, dtype=torch.long, device=device)
    returns_tensor = torch.tensor(returns, dtype=torch.float32, device=device)
    old_log_probs_tensor = torch.stack(old_log_probs).squeeze(-1).to(device)

    # advantage = return - old value
    old_values_tensor = torch.cat(values, dim=0).to(device)
    advantages = returns_tensor - old_values_tensor
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

    # PPO 会对同一批数据训练多轮
    for _ in range(ppo_epochs):
        logits, values_pred = model(states_tensor)
        dist = Categorical(logits=logits)

        new_log_probs = dist.log_prob(actions_tensor)
        entropy = dist.entropy().mean()
        values_pred = values_pred.squeeze(1)

        # 新旧策略概率比
        ratios = torch.exp(new_log_probs - old_log_probs_tensor)

        # PPO 核心：一个正常更新项，一个 clip 后的保守更新项
        surr1 = ratios * advantages
        surr2 = torch.clamp(ratios, 1 - clip_eps, 1 + clip_eps) * advantages
        policy_loss = -torch.min(surr1, surr2).mean()

        value_loss = nn.functional.mse_loss(values_pred, returns_tensor)

        total_loss = policy_loss + value_coef * value_loss - entropy_coef * entropy

        optimizer.zero_grad()
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
        optimizer.step()

    episode_rewards.append(total_reward)
    policy_loss_history.append(float(policy_loss.item()))
    value_loss_history.append(float(value_loss.item()))
    entropy_history.append(float(entropy.item()))

print('训练完成。')
print('最后 20 轮平均 reward:', round(float(np.mean(episode_rewards[-20:])), 2))


## 5. 看训练曲线

这节课你会继续看到：
- reward
- policy loss
- value loss
- entropy

但这次它们背后已经是 PPO 的 clipped objective 在起作用了。


In [ ]:
window = 10
smoothed_rewards = np.convolve(episode_rewards, np.ones(window) / window, mode='valid')

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

axes[0, 0].plot(episode_rewards, color='#1f77b4', alpha=0.5, label='raw')
axes[0, 0].plot(range(window - 1, len(episode_rewards)), smoothed_rewards, color='#d62728', label='smoothed')
axes[0, 0].set_title('每轮 reward')
axes[0, 0].set_xlabel('Episode')
axes[0, 0].set_ylabel('Reward')
axes[0, 0].legend()

axes[0, 1].plot(policy_loss_history, color='#2ca02c', alpha=0.7)
axes[0, 1].set_title('Policy Loss')
axes[0, 1].set_xlabel('Episode')
axes[0, 1].set_ylabel('Loss')

axes[1, 0].plot(value_loss_history, color='#ff7f0e', alpha=0.7)
axes[1, 0].set_title('Value Loss')
axes[1, 0].set_xlabel('Episode')
axes[1, 0].set_ylabel('Loss')

axes[1, 1].plot(entropy_history, color='#9467bd', alpha=0.7)
axes[1, 1].set_title('Entropy')
axes[1, 1].set_xlabel('Episode')
axes[1, 1].set_ylabel('Entropy')

plt.tight_layout()
plt.show()


## 6. 多次测试平均表现

测试时我们继续直接选当前概率最大的动作，方便看训练后的策略质量。


In [ ]:
test_env = gym.make('CartPole-v1')
test_rewards = []

model.eval()
with torch.no_grad():
    for episode_idx in range(5):
        state, info = test_env.reset(seed=123 + episode_idx)
        test_reward = 0.0

        for step in range(500):
            state_tensor = to_tensor(state, device).unsqueeze(0)
            logits, value = model(state_tensor)
            action = int(torch.argmax(logits, dim=1).item())

            state, reward, terminated, truncated, info = test_env.step(action)
            test_reward += reward
            if terminated or truncated:
                break

        test_rewards.append(test_reward)

print('测试 rewards:', test_rewards)
print('测试平均 reward:', round(float(np.mean(test_rewards)), 2))
test_env.close()


## 7. 这节课最该记住什么

如果你想抓住 PPO 最重要的一句话，那就是：

**PPO 用 clipping 限制策略更新幅度，让每次更新既能进步，又不至于把策略改坏。**

这也是它为什么在实际应用里这么常见。


## 8. 下一课最自然学什么

学完 PPO 后，下一步常见的方向有：
- 连续动作版本的 PPO
- 更复杂环境里的 PPO
- SAC / DDPG / TD3 这类连续控制算法

如果按学习路线，我建议下一课先上“连续动作空间是什么，以及为什么 CartPole 这种方法还不够”。
